# Imports

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import FunctionTransformer

# Load Dataset

In [11]:
# Load Titanic sample dataset
data = fetch_openml(name='titanic', version=1, as_frame=True)
df = data.frame
df

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1304,3,0,"Zabour, Miss. Hileni",female,14.5000,1,0,2665,14.4542,NaN,C,NaN,328.0,NaN
1305,3,0,"Zabour, Miss. Thamine",female,NaN,1,0,2665,14.4542,NaN,C,NaN,NaN,NaN
1306,3,0,"Zakarian, Mr. Mapriededer",male,26.5000,0,0,2656,7.2250,NaN,C,NaN,304.0,NaN
1307,3,0,"Zakarian, Mr. Ortin",male,27.0000,0,0,2670,7.2250,NaN,C,NaN,NaN,NaN


In [4]:
df.describe()

,pclass,age,sibsp,parch,fare,body
count,1309.000000,1046.000000,1309.000000,1309.000000,1308.000000,121.000000
mean,2.294882,29.881135,0.498854,0.385027,33.295479,160.809917
std,0.837836,14.413500,1.041658,0.865560,51.758668,97.696922
min,1.000000,0.166700,0.000000,0.000000,0.000000,1.000000
25%,2.000000,21.000000,0.000000,0.000000,7.895800,72.000000
50%,3.000000,28.000000,0.000000,0.000000,14.454200,155.000000
75%,3.000000,39.000000,1.000000,0.000000,31.275000,256.000000
max,3.000000,80.000000,8.000000,9.000000,512.329200,328.000000


# Structure Dataset

In [5]:
# Separate features and target
X = df.drop(columns='survived')
y = df['survived']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Separate numeric and categorical columns
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns

# Define Pipeline 
(and demonstrate single parts of it)
### Pipeline for numeric features (except "body")

In [6]:
# Define preprocessing for numeric features (except "body")
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  # Handle missing values
    ('scaler', StandardScaler())                   # Standardize features
])

In [7]:
numeric_features_wo_body = [feature for feature in numeric_features if feature != 'body']


In [8]:
#Demonstrate application of numeric_transformer (Part1)
X_train_numeric = X_train[numeric_features_wo_body]
X_train_numeric

,pclass,age,sibsp,parch,fare
772,3,17.0,0,0,7.8958
543,2,36.0,0,0,10.5000
289,1,18.0,0,2,79.6500
10,1,47.0,1,0,227.5250
147,1,NaN,0,0,42.4000
...,...,...,...,...,...
1095,3,NaN,0,0,7.6292
1130,3,18.0,0,0,7.7750
1294,3,28.5,0,0,16.1000
860,3,26.0,0,0,7.9250


In [9]:
#Demonstrate application of numeric_transformer (Part1)
numeric_transformer.fit(X_train_numeric)
X_train_numeric = numeric_transformer.transform(X_train_numeric)
X_train_numeric_transformed=pd.DataFrame(X_train_numeric, columns=numeric_features_wo_body)
X_train_numeric_transformed

,pclass,age,sibsp,parch,fare
0,0.840359,-0.956568,-0.495964,-0.442432,-0.495198
1,-0.355097,0.529551,-0.495964,-0.442432,-0.444889
2,-1.550554,-0.878352,-0.495964,1.795376,0.890994
3,-1.550554,1.389936,0.456833,-0.442432,3.747734
4,-1.550554,-0.096184,-0.495964,-0.442432,0.171375
...,...,...,...,...,...
1042,0.840359,-0.096184,-0.495964,-0.442432,-0.500348
1043,0.840359,-0.878352,-0.495964,-0.442432,-0.497532
1044,0.840359,-0.057075,-0.495964,-0.442432,-0.336704
1045,0.840359,-0.252617,-0.495964,-0.442432,-0.494634


In [10]:
def mv_handling_body(X):
    X = X.copy()
    X = X.fillna(0)
    return X


### Pipeline for numeric feature "body"

In [12]:
# Define preprocessing for "body"
body_transformer = Pipeline(steps=[
    ('imputer', FunctionTransformer(mv_handling_body)),  # Handle missing values
    ('scaler', StandardScaler())                                    # Standardize features
])


In [13]:
#Demonstrate application of body_transformer
X_train_body = X_train[['body']]
X_train_body


,body
772,NaN
543,NaN
289,NaN
10,124.0
147,NaN
...,...
1095,NaN
1130,NaN
1294,14.0
860,NaN


In [14]:
#Demonstrate application of body_transformer
body_transformer.fit(X_train_body)
X_train_body = body_transformer.transform(X_train_body)
X_train_body_transformed=pd.DataFrame(X_train_body, columns=['body'])
X_train_body_transformed


,body
0,-0.272292
1,-0.272292
2,-0.272292
3,1.947005
4,-0.272292
...,...
1042,-0.272292
1043,-0.272292
1044,-0.021727
1045,-0.272292


### Pipeline for categoric features

In [15]:
# Define preprocessing for categorical features
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  # Handle missing values
    ('onehot', OneHotEncoder(handle_unknown='ignore'))     # One-hot encoding
])

In [16]:
#Demonstrate application of categorical_transformer (Part1)
X_train_categorical = X_train[categorical_features]
X_train_categorical


,name,sex,ticket,cabin,embarked,boat,home.dest
772,"Dika, Mr. Mirko",male,349232,NaN,S,NaN,NaN
543,"Reeves, Mr. David",male,C.A. 17248,NaN,S,NaN,"Brighton, Sussex"
289,"Taussig, Miss. Ruth",female,110413,E68,S,8,"New York, NY"
10,"Astor, Col. John Jacob",male,PC 17757,C62 C64,C,NaN,"New York, NY"
147,"Harrington, Mr. Charles H",male,113796,NaN,S,NaN,NaN
...,...,...,...,...,...,...,...
1095,"O'Sullivan, Miss. Bridget Mary",female,330909,NaN,Q,NaN,NaN
1130,"Pettersson, Miss. Ellen Natalia",female,347087,NaN,S,NaN,NaN
1294,"Williams, Mr. Leslie",male,54636,NaN,S,NaN,NaN
860,"Heikkinen, Miss. Laina",female,STON/O2. 3101282,NaN,S,NaN,NaN


In [17]:
#Demonstrate application of categorical_transformer (Part2)
categorical_transformer.fit(X_train_categorical)
X_train_categorical = categorical_transformer.transform(X_train_categorical)
X_train_categorical_transformed=pd.DataFrame(X_train_categorical)
X_train_categorical_transformed


,0
0,<Compressed Sparse Row sparse matrix of dtype ...
1,<Compressed Sparse Row sparse matrix of dtype ...
2,<Compressed Sparse Row sparse matrix of dtype ...
3,<Compressed Sparse Row sparse matrix of dtype ...
4,<Compressed Sparse Row sparse matrix of dtype ...
...,...
1042,<Compressed Sparse Row sparse matrix of dtype ...
1043,<Compressed Sparse Row sparse matrix of dtype ...
1044,<Compressed Sparse Row sparse matrix of dtype ...
1045,<Compressed Sparse Row sparse matrix of dtype ...


In [28]:
print(X_train_categorical_transformed.shape)
X_train_categorical_transformed[0:5]  # Display first 5 rows of transformed categorical features
#de-compress the sparse matrix
X_train_categorical_transformed = pd.DataFrame(X_train_categorical.toarray(), columns=categorical_transformer.get_feature_names_out(categorical_features))
X_train_categorical_transformed

(1047, 1)


,"name_Abbing, Mr. Anthony","name_Abbott, Master. Eugene Joseph","name_Abbott, Mr. Rossmore Edward","name_Abbott, Mrs. Stanton (Rosa Hunt)","name_Abelseth, Miss. Karen Marie","name_Abelseth, Mr. Olaus Jorgensen","name_Abelson, Mr. Samuel","name_Abelson, Mrs. Samuel (Hannah Wizosky)","name_Abrahamsson, Mr. Abraham August Johannes","name_Abrahim, Mrs. Joseph (Sophie Halaut Easu)",...,"home.dest_Wimbledon Park, London / Hayling Island, Hants","home.dest_Windsor, England New York, NY","home.dest_Winnipeg, MB","home.dest_Winnipeg, MN","home.dest_Woodford County, KY","home.dest_Worcester, England","home.dest_Worcester, MA","home.dest_Yoevil, England / Cottage Grove, OR","home.dest_Youngstown, OH","home.dest_Zurich, Switzerland"
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1042,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1043,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1044,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1045,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [40]:
#drop name, ticket, cabin, home.dest from categorical features
categorical_features_wo_name_ticket_cabin = [feature for feature in categorical_features if feature not in ['name', 'ticket', 'cabin','home.dest']]
X_train_categorical = X_train[categorical_features_wo_name_ticket_cabin]
categorical_transformer.fit(X_train_categorical)
X_train_categorical = categorical_transformer.transform(X_train_categorical)

#de-compress the sparse matrix
X_train_categorical_transformed = pd.DataFrame(X_train_categorical.toarray(), columns=categorical_transformer.get_feature_names_out(categorical_features_wo_name_ticket_cabin))
X_train_categorical_transformed

,sex_female,sex_male,embarked_C,embarked_Q,embarked_S,boat_1,boat_10,boat_11,boat_12,boat_13,...,boat_6,boat_7,boat_8,boat_8 10,boat_9,boat_A,boat_B,boat_C,boat_C D,boat_D
0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1042,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1043,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1044,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1045,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [41]:
# Combine preprocessors in a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features_wo_body),
        ('body', body_transformer, ['body']),
        ('cat', categorical_transformer, categorical_features_wo_name_ticket_cabin)
    ]
)

In [42]:
# Define the final pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))  # Classifier
])

# Train Pipeline

In [43]:
# Train the pipeline
pipeline.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['pclass', 'age', 'sibsp',
                                                   'parch', 'fare']),
                                                 ('body',
                                                  Pipeline(steps=[('imputer',
                                                                   FunctionTransformer(func=<function mv_handling_body at 0x0000025D7B5A4F40>)),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['body']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['sex', 'embarked',
                                                   'boat'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

# Evaluate Pipeline

In [44]:
# Evaluate the pipeline
train_score = pipeline.score(X_train, y_train)
test_score = pipeline.score(X_test, y_test)

print(f"Training accuracy: {train_score:.2f}")
print(f"Testing accuracy: {test_score:.2f}")


Training accuracy: 0.99
Testing accuracy: 0.95
